# Módulo 04 · Aula 01 — Funções Avançadas

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O script virou um monstro de 800 linhas. Toda vez que preciso adicionar uma métrica nova, copio um bloco de 40 linhas e mudo três palavras. Já tem cinco cópias quase iguais. Ontem corrigi um bug em uma delas e esqueci das outras quatro."*
> — Você, olhando o próprio código

No Módulo 01 você aprendeu que função é reuso. Agora vai aprender que **função é um valor** — e que isso abre um repertório inteiro de soluções para o problema acima.

## O que você vai aprender aqui

| # | Tópico | Resolve |
|---|--------|---------|
| 1 | `*args` e `**kwargs` | Funções que aceitam qualquer assinatura |
| 2 | Desempacotamento na chamada | Repassar argumentos sem reescrever |
| 3 | Funções como valores | Guardar comportamento em variáveis e dicionários |
| 4 | `lambda` | Funções descartáveis de uma linha |
| 5 | `map`, `filter`, `sorted`, `reduce` | Programação funcional em Python |
| 6 | **Closures** | Funções que lembram do contexto |
| 7 | Fábricas de função | Gerar funções sob medida |
| 8 | **Decoradores** | Adicionar comportamento sem tocar no código |
| 9 | `functools` | `wraps`, `lru_cache`, `partial` |

## 1. `*args` e `**kwargs`

Duas sintaxes que resolvem o problema "não sei quantos argumentos vou receber".

| Sintaxe | Coleta | Vira |
|---------|--------|------|
| `*args` | Argumentos **posicionais** extras | Uma **tupla** |
| `**kwargs` | Argumentos **nomeados** extras | Um **dicionário** |

> 💡 Os nomes `args` e `kwargs` são apenas convenção. O que importa são os asteriscos: `*` empacota posicionais, `**` empacota nomeados. Você poderia escrever `*valores` e `**opcoes` — e às vezes é mais legível fazer isso.

In [ ]:
def somar_tudo(*numeros):
    """Aceita qualquer quantidade de números."""
    print(f"  recebi: {numeros}  (tipo: {type(numeros).__name__})")
    return sum(numeros)


print(somar_tudo(1, 2, 3))
print(somar_tudo(10, 20, 30, 40, 50))
print(somar_tudo())          # tupla vazia — sum(()) é 0

In [ ]:
def configurar(**opcoes):
    """Aceita qualquer quantidade de argumentos nomeados."""
    print(f"  recebi: {opcoes}  (tipo: {type(opcoes).__name__})")
    for chave, valor in opcoes.items():
        print(f"    {chave} = {valor}")


configurar(host="localhost", porta=5432, debug=True)

### A ordem completa dos parâmetros

Python permite combinar tudo, mas **a ordem é obrigatória**:

```python
def f(posicional, padrao=1, *args, so_nomeado, com_padrao=2, **kwargs):
    ...
```

| Posição | Tipo |
|---------|------|
| 1 | Posicionais obrigatórios |
| 2 | Posicionais com padrão |
| 3 | `*args` |
| 4 | **Só-nomeados** (tudo depois do `*`) |
| 5 | `**kwargs` |

In [ ]:
def registrar_venda(produto, quantidade=1, *extras, canal, desconto=0.0, **metadados):
    """Assinatura completa, para você ver cada parte funcionando."""
    print(f"produto   : {produto}")
    print(f"quantidade: {quantidade}")
    print(f"extras    : {extras}       <- *args")
    print(f"canal     : {canal}        <- só-nomeado (obrigatório!)")
    print(f"desconto  : {desconto}")
    print(f"metadados : {metadados}    <- **kwargs")


registrar_venda(
    "Notebook", 2, "brinde", "embalagem-presente",
    canal="site", desconto=0.1, cupom="AURORA10", vendedor="Ana",
)

In [ ]:
# O * sozinho força tudo depois dele a ser nomeado
def gerar_relatorio(dados, *, formato="texto", incluir_total=True):
    return f"[{formato}] {len(dados)} registros, total={incluir_total}"


print(gerar_relatorio([1, 2, 3], formato="json"))

# gerar_relatorio([1,2,3], "json")   # <- descomente: TypeError

> 💡 **Por que forçar nomeado?** Porque `processar(dados, True, False, 3)` é ilegível. Seis meses depois, ninguém sabe o que cada booleano significa — nem você. `processar(dados, validar=True, verbose=False, tentativas=3)` se explica sozinho.
>
> **Regra prática:** todo parâmetro booleano e todo "número mágico" deveria ser só-nomeado.

### Desempacotamento na **chamada**

Os mesmos operadores funcionam ao contrário: **espalhando** uma coleção nos argumentos.

In [ ]:
def calcular_total(quantidade, preco, desconto):
    return round(quantidade * preco * (1 - desconto), 2)


valores = (3, 249.00, 0.10)
print(calcular_total(*valores))          # espalha a tupla

parametros = {"quantidade": 2, "preco": 2599.90, "desconto": 0.05}
print(calcular_total(**parametros))      # espalha o dicionário

In [ ]:
# O uso mais valioso: repassar argumentos sem conhecê-los
def com_log(funcao, *args, **kwargs):
    """Executa qualquer função, registrando entrada e saída."""
    nome = funcao.__name__
    print(f"→ {nome}(args={args}, kwargs={kwargs})")
    resultado = funcao(*args, **kwargs)
    print(f"← {nome} devolveu {resultado}")
    return resultado


com_log(calcular_total, 3, 249.00, desconto=0.10)
print()
com_log(somar_tudo, 1, 2, 3, 4, 5)

> 💭 **Guarde este padrão.** `funcao(*args, **kwargs)` é o mecanismo que faz decoradores existirem — e você vai chegar lá na seção 8.

## 2. Funções são valores

Em Python, uma função é um **objeto** como qualquer outro. Ela pode ser:

- atribuída a uma variável
- guardada em lista, dicionário ou conjunto
- passada como argumento
- devolvida por outra função

Isso se chama *funções de primeira classe*, e é a base de tudo nesta aula.

> ⚠️ **`funcao` vs `funcao()`.** Sem parênteses você referencia o objeto. Com parênteses você o **executa**. Confundir os dois é o erro nº 1 desta aula.

In [ ]:
def dobrar(x):
    return x * 2


print("dobrar   :", dobrar)      # o objeto função
print("dobrar(5):", dobrar(5))   # o resultado da chamada
print()

# Atribuindo a outro nome — não copia, só cria outro rótulo (lembra da aula 01_01?)
duplicar = dobrar
print(duplicar(21))
print("Mesmo objeto?", duplicar is dobrar)
print("__name__ continua sendo:", duplicar.__name__)

In [ ]:
# Funções têm atributos, como qualquer objeto
def calcular_frete(valor, uf):
    """Calcula o frete conforme a regra da Aurora."""
    return 0.0 if valor >= 500 else (12.90 if uf == "SP" else 29.90)


print("nome    :", calcular_frete.__name__)
print("doc     :", calcular_frete.__doc__)
print("módulo  :", calcular_frete.__module__)
print("assinatura:", calcular_frete.__code__.co_varnames[:2])

# Você pode até anexar atributos
calcular_frete.versao = "2.0"
print("versão  :", calcular_frete.versao)

### O padrão do dicionário de despacho

Este é o primeiro remédio contra o "monstro de 800 linhas": em vez de uma cadeia gigante de `if/elif`, um **dicionário que mapeia nome → função**.

In [ ]:
# ❌ ANTES: a cadeia que cresce para sempre
def relatorio_antigo(tipo, dados):
    if tipo == "cidade":
        return f"[cidade] {len(dados)} registros"
    elif tipo == "produto":
        return f"[produto] {len(dados)} registros"
    elif tipo == "canal":
        return f"[canal] {len(dados)} registros"
    elif tipo == "cliente":
        return f"[cliente] {len(dados)} registros"
    else:
        raise ValueError(f"tipo desconhecido: {tipo}")


print(relatorio_antigo("cidade", [1, 2, 3]))

In [ ]:
# ✅ DEPOIS: dicionário de despacho
def por_cidade(dados):
    return f"[cidade] {len(dados)} registros"


def por_produto(dados):
    return f"[produto] {len(dados)} registros"


def por_canal(dados):
    return f"[canal] {len(dados)} registros"


RELATORIOS = {
    "cidade": por_cidade,
    "produto": por_produto,
    "canal": por_canal,
}


def gerar(tipo, dados):
    funcao = RELATORIOS.get(tipo)
    if funcao is None:
        raise ValueError(f"tipo desconhecido: {tipo}. Disponíveis: {sorted(RELATORIOS)}")
    return funcao(dados)


for t in ["cidade", "produto", "canal"]:
    print(gerar(t, [1, 2, 3]))

print("\nRelatórios disponíveis:", sorted(RELATORIOS))

> 💡 **Três ganhos de uma vez:**
>
> 1. **Adicionar um relatório novo** é adicionar uma linha no dicionário — não editar uma função existente. (Isso tem nome: *princípio aberto-fechado*.)
> 2. **A lista de opções vira dado.** `sorted(RELATORIOS)` alimenta o `--help` da CLI automaticamente.
> 3. **A busca é O(1)**, não O(n) como a cadeia de `elif`.
>
> Você já viu isso no M03: `RELATORIOS` em `relatorios_sql.py` é exatamente este padrão.

## 3. `lambda` — funções anônimas

```python
lambda parametros: expressao
```

Uma `lambda` é uma função de **uma expressão só**, sem nome. Ela **devolve** o valor da expressão automaticamente — não tem `return`.

In [ ]:
dobrar_lambda = lambda x: x * 2
print(dobrar_lambda(21))

# Equivalente exato:
def dobrar_def(x):
    return x * 2

print(dobrar_def(21))
print("Mesmo tipo?", type(dobrar_lambda) is type(dobrar_def))

### Quando usar (e quando não)

| ✅ Use `lambda` | ❌ Use `def` |
|-----------------|-------------|
| Como `key=` em `sorted`/`min`/`max` | A lógica tem mais de uma expressão |
| Callback curto passado a outra função | Precisa de docstring |
| Valor padrão de `defaultdict` | Vai ser reusada em vários lugares |
| | Precisa de nome no traceback |

> 🔴 **Nunca atribua uma `lambda` a um nome.** Se você escreveu `f = lambda x: ...`, use `def f(x):` — é a mesma coisa, mas com nome real no traceback e espaço para docstring. A [PEP 8](https://peps.python.org/pep-0008/#programming-recommendations) diz isso explicitamente.

In [ ]:
produtos = [
    {"nome": "Notebook Dell",  "preco": 2599.90, "estoque": 14, "categoria": "Notebooks"},
    {"nome": "Mouse Logitech", "preco": 89.90,   "estoque": 240, "categoria": "Periféricos"},
    {"nome": "Monitor LG",     "preco": 1199.00, "estoque": 31, "categoria": "Monitores"},
    {"nome": "Teclado Redragon","preco": 249.00, "estoque": 64, "categoria": "Periféricos"},
    {"nome": "SSD 1TB",        "preco": 489.00,  "estoque": 52, "categoria": "Armazenamento"},
]

print("Por preço:")
for p in sorted(produtos, key=lambda p: p["preco"]):
    print(f"  {p['nome']:<20} R$ {p['preco']:>9,.2f}")

print("\nMais caro:", max(produtos, key=lambda p: p["preco"])["nome"])
print("Menor estoque:", min(produtos, key=lambda p: p["estoque"])["nome"])

In [ ]:
# Ordenação por múltiplos critérios: devolva uma TUPLA
print("Por categoria, e dentro dela por preço decrescente:")
for p in sorted(produtos, key=lambda p: (p["categoria"], -p["preco"])):
    print(f"  {p['categoria']:<16} {p['nome']:<20} R$ {p['preco']:>9,.2f}")

> 💡 **O truque do `-preco`:** `sorted` não aceita direção diferente por campo. Para ordenar um campo crescente e outro decrescente, negue o numérico. Para texto, você precisaria de dois `sorted` encadeados (a ordenação do Python é *estável*, então funciona).

## 4. `map`, `filter` e `reduce`

Três funções que aplicam uma função a uma coleção.

| Função | Faz | Equivalente em comprehension |
|--------|-----|------------------------------|
| `map(f, xs)` | Aplica `f` a cada elemento | `[f(x) for x in xs]` |
| `filter(f, xs)` | Mantém os que satisfazem `f` | `[x for x in xs if f(x)]` |
| `reduce(f, xs)` | Colapsa tudo em um valor | (não tem — é laço) |

> 🧭 **Em Python, prefira comprehensions.** Elas são mais legíveis para quem lê Python, e o Guido van Rossum (criador da linguagem) chegou a propor remover `map` e `filter` da linguagem. `map` e `filter` continuam úteis quando você **já tem** a função pronta e não precisa de lambda.

In [ ]:
precos = [2599.90, 89.90, 1199.00, 249.00, 489.00]

# map devolve um ITERADOR preguiçoso — precisa de list() para materializar
com_imposto = map(lambda p: round(p * 1.18, 2), precos)
print("O objeto map:", com_imposto)
print("Materializado:", list(com_imposto))
print("De novo:      ", list(com_imposto), "  <- ⚠️ vazio! iterador esgotou")

In [ ]:
# Comparação lado a lado
print("map     :", list(map(lambda p: p * 1.18, precos)))
print("compreh.:", [p * 1.18 for p in precos])
print()
print("filter  :", list(filter(lambda p: p > 500, precos)))
print("compreh.:", [p for p in precos if p > 500])
print()
# Quando map brilha: a função JÁ existe
print("map com função pronta:", list(map(str.upper, ["ana", "bruno", "carla"])))

In [ ]:
from functools import reduce

# reduce: acumula da esquerda para a direita
soma = reduce(lambda acc, x: acc + x, precos)
maior = reduce(lambda acc, x: acc if acc > x else x, precos)

print(f"soma  : {soma:,.2f}")
print(f"maior : {maior:,.2f}")
print()
print("💡 Mas prefira os embutidos, que são mais rápidos e legíveis:")
print(f"sum() : {sum(precos):,.2f}")
print(f"max() : {max(precos):,.2f}")

In [ ]:
# reduce é útil quando NÃO existe um embutido equivalente
pedidos = [
    {"cidade": "Campinas",  "valor": 5199.80},
    {"cidade": "São Paulo", "valor": 899.00},
    {"cidade": "Campinas",  "valor": 1199.00},
    {"cidade": "Sorocaba",  "valor": 249.00},
]


def acumular(acc, pedido):
    acc[pedido["cidade"]] = acc.get(pedido["cidade"], 0) + pedido["valor"]
    return acc


print(reduce(acumular, pedidos, {}))

> ⚠️ **O terceiro argumento do `reduce` é o valor inicial.** Sem ele, `reduce` usa o primeiro elemento como acumulador — e estoura `TypeError` com lista vazia. Com o inicial (`{}` acima), lista vazia devolve o inicial. **Sempre passe o valor inicial.**

## 5. Closures — funções que lembram

Uma **closure** é uma função que "captura" variáveis do escopo onde foi definida, e continua acessando-as depois que aquele escopo terminou.

```
def externa(x):          ← o escopo "enclosing"
    def interna():
        return x         ← captura o x
    return interna       ← devolve a função, e o x vai junto
```

Revise o **LEGB** da aula 01_04: o `E` de *Enclosing* é exatamente isto.

In [ ]:
def criar_saudacao(saudacao):
    """Fábrica: devolve uma função personalizada."""
    def cumprimentar(nome):
        return f"{saudacao}, {nome}!"
    return cumprimentar


ola = criar_saudacao("Olá")
bomdia = criar_saudacao("Bom dia")

print(ola("Maria"))
print(bomdia("João"))

# A variável 'saudacao' sumiu do escopo, mas a função lembra dela:
print("\nO que ficou capturado:", ola.__closure__[0].cell_contents)

In [ ]:
# Closure com estado mutável: precisa de nonlocal
def criar_contador(inicio=0):
    contagem = inicio

    def incrementar(passo=1):
        nonlocal contagem      # sem isto, 'contagem' viraria local
        contagem += passo
        return contagem

    def ler():
        return contagem

    incrementar.ler = ler      # anexando função como atributo
    return incrementar


contador = criar_contador(100)
print(contador(), contador(), contador(10))
print("Estado atual:", contador.ler())

# Contadores independentes — cada closure tem seu próprio estado
outro = criar_contador()
print("Contador novo:", outro())

### 🔴 A armadilha da closure em laço

Este bug aparece em toda entrevista técnica — e em código real.

**A closure captura a *variável*, não o *valor*.** Quando o laço termina, todas as funções criadas apontam para a mesma variável, com o valor final.

In [ ]:
# ❌ ERRADO
funcoes_erradas = []
for i in range(3):
    funcoes_erradas.append(lambda: i)

print("Esperado: 0 1 2")
print("Obtido:  ", [f() for f in funcoes_erradas])

In [ ]:
# ✅ CORREÇÃO 1: valor padrão captura o valor NA HORA da definição
funcoes_ok = []
for i in range(3):
    funcoes_ok.append(lambda i=i: i)

print("Com valor padrão:", [f() for f in funcoes_ok])


# ✅ CORREÇÃO 2: uma fábrica cria um escopo novo a cada chamada
def fabricar(valor):
    return lambda: valor

funcoes_ok2 = [fabricar(i) for i in range(3)]
print("Com fábrica:     ", [f() for f in funcoes_ok2])

## 6. Fábricas de função

Uma **fábrica** é uma função que devolve funções configuradas. É o padrão que elimina código quase-duplicado.

In [ ]:
def criar_validador(campo, minimo=None, maximo=None, obrigatorio=True):
    """Gera um validador para um campo específico."""
    def validar(registro):
        valor = registro.get(campo)

        if valor is None:
            return None if not obrigatorio else f"{campo}: obrigatório"
        if minimo is not None and valor < minimo:
            return f"{campo}: {valor} abaixo do mínimo {minimo}"
        if maximo is not None and valor > maximo:
            return f"{campo}: {valor} acima do máximo {maximo}"
        return None

    validar.__name__ = f"validar_{campo}"
    return validar


# Um validador por regra — nenhuma linha duplicada
VALIDADORES = [
    criar_validador("quantidade", minimo=1, maximo=1000),
    criar_validador("preco", minimo=0),
    criar_validador("desconto", minimo=0, maximo=1, obrigatorio=False),
]

registros = [
    {"quantidade": 5,    "preco": 249.00, "desconto": 0.1},
    {"quantidade": 0,    "preco": 249.00},
    {"quantidade": 2000, "preco": -10.00, "desconto": 1.5},
]

for i, reg in enumerate(registros, 1):
    erros = [e for v in VALIDADORES if (e := v(reg)) is not None]
    marca = "OK" if not erros else "ERRO"
    print(f"[{marca}] registro {i}: {erros or 'válido'}")

> 💡 **O `:=` (operador morsa)**, do Python 3.8, atribui **dentro** de uma expressão. Ele permite chamar `v(reg)` uma vez só e usar o resultado no filtro e no valor. Sem ele, você chamaria a função duas vezes ou escreveria um laço explícito.

## 7. Decoradores

Um **decorador** é uma função que recebe uma função e devolve outra função — normalmente uma versão "envelopada" com comportamento extra.

```python
@meu_decorador
def minha_funcao():
    ...

# é exatamente o mesmo que:
minha_funcao = meu_decorador(minha_funcao)
```

O `@` é só açúcar sintático. Não há mágica.

**Para que serve:** adicionar logging, medição de tempo, cache, autenticação, retentativa, validação — **sem tocar no código da função original**.

In [ ]:
def registrar(funcao):
    """Decorador que anuncia entrada e saída."""
    def envelope(*args, **kwargs):
        print(f"→ chamando {funcao.__name__}")
        resultado = funcao(*args, **kwargs)
        print(f"← {funcao.__name__} devolveu {resultado!r}")
        return resultado
    return envelope


@registrar
def calcular_desconto(valor, percentual):
    return round(valor * percentual, 2)


calcular_desconto(1000, 0.15)

### 🔴 O problema: o decorador apaga a identidade da função

In [ ]:
print("nome :", calcular_desconto.__name__)
print("doc  :", calcular_desconto.__doc__)

> 😬 A função virou `envelope`. Isso quebra o `help()`, os tracebacks, a introspecção e várias bibliotecas (inclusive o FastAPI, que você vai usar no M06).
>
> **A correção é uma linha:** `@functools.wraps(funcao)` no envelope. Ela copia `__name__`, `__doc__`, `__module__` e a assinatura da função original.
>
> ⚠️ **Nunca escreva um decorador sem `@wraps`.**

In [ ]:
import functools


def registrar(funcao):
    """Decorador que anuncia entrada e saída — agora preservando a identidade."""
    @functools.wraps(funcao)          # ← a linha que faltava
    def envelope(*args, **kwargs):
        print(f"→ {funcao.__name__}({args}, {kwargs})")
        resultado = funcao(*args, **kwargs)
        print(f"← {resultado!r}")
        return resultado
    return envelope


@registrar
def calcular_desconto(valor, percentual):
    """Calcula o valor do desconto."""
    return round(valor * percentual, 2)


calcular_desconto(1000, 0.15)
print()
print("nome :", calcular_desconto.__name__)
print("doc  :", calcular_desconto.__doc__)
print("original acessível em:", calcular_desconto.__wrapped__.__name__)

In [ ]:
# Decorador útil de verdade: medir tempo
import time


def cronometrar(funcao):
    """Mede e reporta o tempo de execução."""
    @functools.wraps(funcao)
    def envelope(*args, **kwargs):
        inicio = time.perf_counter()
        try:
            return funcao(*args, **kwargs)
        finally:
            # finally garante o registro mesmo se a função levantar exceção
            decorrido = (time.perf_counter() - inicio) * 1000
            print(f"⏱  {funcao.__name__}: {decorrido:.2f} ms")
    return envelope


@cronometrar
def processar_vendas(n):
    """Simula processamento pesado."""
    return sum(i ** 2 for i in range(n))


processar_vendas(100_000)
processar_vendas(1_000_000)

### Decoradores com parâmetros

Para que `@decorador(config)` funcione, você precisa de **três** níveis:

```
def decorador(parametro):        ← 1. recebe a configuração
    def real(funcao):            ← 2. recebe a função
        def envelope(*a, **k):   ← 3. recebe os argumentos da chamada
            ...
        return envelope
    return real
```

Lembre-se: `@decorador(config)` primeiro **chama** `decorador(config)`, e o resultado é que decora a função.

In [ ]:
def tentar_novamente(tentativas=3, espera=0.1, excecoes=(Exception,)):
    """Repete a função em caso de falha."""
    def decorador(funcao):
        @functools.wraps(funcao)
        def envelope(*args, **kwargs):
            ultimo_erro = None
            for tentativa in range(1, tentativas + 1):
                try:
                    return funcao(*args, **kwargs)
                except excecoes as erro:
                    ultimo_erro = erro
                    print(f"  ⚠️  tentativa {tentativa}/{tentativas} falhou: {erro}")
                    if tentativa < tentativas:
                        time.sleep(espera)
            raise ultimo_erro
        return envelope
    return decorador


# Simulando uma API instável
contador_chamadas = {"n": 0}


@tentar_novamente(tentativas=4, espera=0.05, excecoes=(ConnectionError,))
def consultar_transportadora(cep):
    contador_chamadas["n"] += 1
    if contador_chamadas["n"] < 3:
        raise ConnectionError("timeout na transportadora")
    return {"cep": cep, "prazo_dias": 3, "valor": 24.90}


print(consultar_transportadora("13000-000"))

> 💭 **Este decorador é o embrião do que você vai construir no Módulo 07**, quando o Atlas precisar falar com a transportadora e o gateway de pagamento. Lá ele ganha *backoff exponencial* — a espera dobra a cada tentativa, para não sobrecarregar um serviço que já está sofrendo.

In [ ]:
# Empilhando decoradores: aplicam-se de BAIXO para CIMA
@registrar
@cronometrar
def calcular_faturamento(vendas):
    """Soma o valor das vendas pagas."""
    return sum(v["valor"] for v in vendas if v["status"] == "pago")


vendas = [
    {"valor": 5199.80, "status": "pago"},
    {"valor": 899.00,  "status": "cancelado"},
    {"valor": 1199.00, "status": "pago"},
]

calcular_faturamento(vendas)

> 💡 **A ordem importa.** `@registrar` acima de `@cronometrar` significa `registrar(cronometrar(funcao))`. O `cronometrar` fica mais perto da função original, então mede só a execução dela; o `registrar` envolve tudo por fora.
>
> Se você inverter, o cronômetro passa a medir também o tempo dos `print` do log.

## 8. `functools` — o kit essencial

| Ferramenta | Faz |
|------------|-----|
| `@wraps` | Preserva a identidade da função decorada |
| `@lru_cache` | Memoiza o resultado — cache automático |
| `@cache` | `lru_cache(maxsize=None)`, Python 3.9+ |
| `partial` | Fixa alguns argumentos e devolve uma função nova |
| `reduce` | Acumulação |
| `@singledispatch` | Sobrecarga por tipo do primeiro argumento |

In [ ]:
# lru_cache: transforma recursão exponencial em linear
@functools.lru_cache(maxsize=None)
def fibonacci(n):
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)


inicio = time.perf_counter()
resultado = fibonacci(35)
print(f"fib(35) = {resultado:,}  em {(time.perf_counter()-inicio)*1000:.2f} ms")
print("Estatísticas do cache:", fibonacci.cache_info())

> ⚠️ **`lru_cache` só funciona se os argumentos forem hasheáveis** (`int`, `str`, `tuple`...). Passar uma `list` ou `dict` levanta `TypeError`.
>
> 🔴 **E só use em funções puras.** Se a função lê um arquivo, consulta o banco ou depende da hora atual, o cache vai devolver dado velho e você vai perder horas procurando o bug. Cache é para cálculo determinístico.

In [ ]:
from functools import partial

# partial: fixa argumentos e devolve uma função nova
def formatar(valor, moeda="R$", casas=2, separador_milhar=True):
    texto = f"{valor:,.{casas}f}" if separador_milhar else f"{valor:.{casas}f}"
    return f"{moeda} {texto}"


formatar_brl = partial(formatar, moeda="R$")
formatar_usd = partial(formatar, moeda="US$")
formatar_inteiro = partial(formatar, casas=0)

print(formatar_brl(1234.5))
print(formatar_usd(1234.5))
print(formatar_inteiro(1234.5))

# Muito usado como key= quando a função pede um argumento extra
def por_campo(registro, campo):
    return registro[campo]

print("\nOrdenado por estoque:",
      [p["nome"] for p in sorted(produtos, key=partial(por_campo, campo="estoque"))])

## 🔧 Prática guiada — Um motor de regras para a Aurora

Vamos aplicar tudo: dicionário de despacho, fábricas, closures e decoradores para construir um **motor de precificação** extensível.

Compare mentalmente com a alternativa: um `if/elif` de 200 linhas em que adicionar uma regra exige editar a função existente.

In [ ]:
import functools
import time

# ═══════════════════════════════════════════════════════════════
#  Infraestrutura: um registro de regras alimentado por decorador
# ═══════════════════════════════════════════════════════════════

REGRAS = {}


def regra(nome, prioridade=100):
    """Decorador que REGISTRA a função no motor de regras.

    Repare: o decorador não altera a função — ele a cataloga.
    Esse é um uso de decorador que muita gente não conhece.
    """
    def decorador(funcao):
        REGRAS[nome] = {"funcao": funcao, "prioridade": prioridade, "nome": nome}
        return funcao          # devolve a função INTACTA
    return decorador


def aplicar_regras(pedido, teto=0.25):
    """Aplica todas as regras registradas, respeitando o teto de desconto."""
    ajustes = []
    for info in sorted(REGRAS.values(), key=lambda r: r["prioridade"]):
        resultado = info["funcao"](pedido)
        if resultado:
            ajustes.append((info["nome"], resultado))

    total = sum(a for _, a in ajustes)
    total_limitado = max(-teto, min(teto, total))
    return {"ajustes": ajustes, "ajuste_bruto": total, "ajuste_aplicado": total_limitado}


print("✅ Motor pronto. Nenhuma regra registrada ainda:", REGRAS)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  As regras de negócio. Adicionar uma nova = escrever uma função.
#  Nenhum código existente é tocado.
# ═══════════════════════════════════════════════════════════════

@regra("volume", prioridade=10)
def regra_volume(pedido):
    """10 unidades ou mais: 8% de desconto."""
    return -0.08 if pedido["quantidade"] >= 10 else 0


@regra("categoria_monitores", prioridade=20)
def regra_monitores(pedido):
    """Queima de estoque de monitores: 5%."""
    return -0.05 if pedido["categoria"] == "Monitores" else 0


@regra("cliente_fiel", prioridade=30)
def regra_fidelidade(pedido):
    """Cliente com 3+ pedidos: 7%."""
    return -0.07 if pedido.get("pedidos_anteriores", 0) >= 3 else 0


@regra("comissao_marketplace", prioridade=40)
def regra_marketplace(pedido):
    """Marketplace cobra comissão — repassamos 12%."""
    return +0.12 if pedido["canal"] == "marketplace" else 0


@regra("alto_valor", prioridade=50)
def regra_alto_valor(pedido):
    """Pedido acima de R$ 5.000: 10%."""
    return -0.10 if pedido["quantidade"] * pedido["preco"] >= 5000 else 0


print(f"✅ {len(REGRAS)} regras registradas:", sorted(REGRAS))

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Executando
# ═══════════════════════════════════════════════════════════════

pedidos_teste = [
    {"produto": "Mouse Logitech", "categoria": "Periféricos", "quantidade": 12,
     "preco": 89.90, "canal": "site", "pedidos_anteriores": 5},
    {"produto": "Monitor LG", "categoria": "Monitores", "quantidade": 1,
     "preco": 1199.00, "canal": "marketplace", "pedidos_anteriores": 0},
    {"produto": "Notebook Dell", "categoria": "Notebooks", "quantidade": 3,
     "preco": 2599.90, "canal": "app", "pedidos_anteriores": 8},
]

for pedido in pedidos_teste:
    bruto = pedido["quantidade"] * pedido["preco"]
    r = aplicar_regras(pedido)
    final = bruto * (1 + r["ajuste_aplicado"])

    print(f"\n{'─' * 62}")
    print(f"{pedido['produto']} x{pedido['quantidade']} via {pedido['canal']}")
    print(f"{'Bruto':<28}{bruto:>16,.2f}")
    for nome, ajuste in r["ajustes"]:
        print(f"  {nome:<26}{ajuste:>+15.1%}")
    if abs(r['ajuste_bruto'] - r['ajuste_aplicado']) > 1e-9:
        print(f"  {'(teto aplicado)':<26}{r['ajuste_bruto']:>+15.1%} → {r['ajuste_aplicado']:+.1%}")
    print(f"{'Ajuste final':<28}{r['ajuste_aplicado']:>+15.1%}")
    print(f"{'TOTAL':<28}{final:>16,.2f}")

In [ ]:
# Adicionando uma regra NOVA sem tocar em nada do que já existe
@regra("black_friday", prioridade=5)
def regra_black_friday(pedido):
    """Promoção sazonal: 15% em tudo."""
    return -0.15 if pedido.get("data", "").startswith("2026-11") else 0


# Este pedido dispara VÁRIAS regras ao mesmo tempo — e estoura o teto
pedido_bf = {"produto": "SSD 1TB", "categoria": "Armazenamento", "quantidade": 12,
             "preco": 489.00, "canal": "site", "pedidos_anteriores": 6,
             "data": "2026-11-28"}

r = aplicar_regras(pedido_bf)
print(f"Regras registradas agora: {len(REGRAS)}\n")
for nome, ajuste in r["ajustes"]:
    print(f"  {nome:<26}{ajuste:>+10.1%}")
print(f"  {'-' * 36}")
print(f"  {'soma dos ajustes':<26}{r['ajuste_bruto']:>+10.1%}")
print(f"  {'TETO aplicado (25%)':<26}{r['ajuste_aplicado']:>+10.1%}  ← a trava funcionou")

> 💭 **Volte à dor do início da aula.** "Copio um bloco de 40 linhas e mudo três palavras."
>
> Neste motor, adicionar uma regra é escrever **uma função de 3 linhas**. Nenhum código existente foi editado — logo, nenhum bug foi introduzido no que já funcionava. O `REGRAS` sabe listar tudo que existe. E o teto de desconto está em **um** lugar.
>
> Isso é o que "código extensível" significa na prática.

## 📝 Exercícios

**E1.** Escreva `estatisticas(*valores, **opcoes)` que aceite qualquer quantidade de números e as opções `casas` (padrão 2) e `incluir_mediana` (padrão False). Devolva um dict.

**E2.** Escreva `mesclar_configs(*configs)` que receba vários dicionários e devolva um só, com os últimos sobrescrevendo os primeiros.

**E3.** Crie um dicionário de despacho `FORMATADORES` mapeando `"brl"`, `"usd"`, `"pct"`, `"int"` para funções de formatação. Escreva `formatar(valor, tipo)` que use o dicionário e dê erro claro para tipo desconhecido.

**E4.** Ordene a lista `produtos` (definida acima) por: (a) categoria e depois preço decrescente; (b) comprimento do nome; (c) valor em estoque (`preco * estoque`) decrescente.

**E5.** Explique por que o código abaixo imprime `[2, 2, 2]` e corrija de **duas** formas:
```python
fs = []
for i in range(3):
    fs.append(lambda: i)
print([f() for f in fs])
```

**E6.** Escreva `criar_acumulador()` que devolva uma função com estado interno: cada chamada soma o valor recebido e devolve o total acumulado. Crie dois acumuladores e prove que são independentes.

**E7.** Escreva um decorador `@contar_chamadas` que registre quantas vezes a função foi chamada, acessível em `funcao.chamadas`. Use `@wraps`.

**E8.** Escreva um decorador `@validar_tipos` que verifique os argumentos contra as anotações de tipo da função e levante `TypeError` se não baterem. *(Dica: `funcao.__annotations__` e `inspect.signature`.)*

**E9.** Escreva `@limitar_taxa(por_segundo=2)` que impeça a função de ser chamada mais de N vezes por segundo, esperando quando necessário.

**E10.** Use `functools.partial` para criar três versões especializadas de uma função `conectar(host, porta, timeout, ssl)`: `conectar_local`, `conectar_producao` e `conectar_teste`.

**E11.** Escreva um decorador `@memoizar` do zero (sem usar `lru_cache`), com dicionário próprio e um método `.limpar_cache()`. Compare a performance com `lru_cache` em `fibonacci(30)`.

**E12.** Estenda o motor de regras da prática guiada com duas regras novas: frete grátis acima de R$ 500 e sobretaxa de 5% para UFs fora do Sudeste. Prove que o código existente não precisou mudar.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

## 📋 Cola de referência

```python
# ── *args e **kwargs ──
def f(obrig, padrao=1, *args, so_nomeado, opc=2, **kwargs): ...
def f(dados, *, validar=True): ...        # tudo após * é só-nomeado

f(*lista)                                  # espalha posicionais
f(**dicionario)                            # espalha nomeados
def envelope(*a, **k): return funcao(*a, **k)   # repasse genérico

# ── Funções como valores ──
DESPACHO = {"a": func_a, "b": func_b}      # em vez de if/elif
DESPACHO["a"](dados)
funcao.__name__  .__doc__  .__wrapped__

# ── lambda ──
sorted(xs, key=lambda x: x["campo"])
sorted(xs, key=lambda x: (x["a"], -x["b"]))    # multi-critério
max(xs, key=lambda x: x["valor"])
# ❌ nunca: f = lambda x: ...   → use def

# ── map / filter / reduce ──
list(map(f, xs))          # ≡ [f(x) for x in xs]
list(filter(f, xs))       # ≡ [x for x in xs if f(x)]
reduce(f, xs, inicial)    # SEMPRE passe o inicial
# ⚠️ map/filter devolvem iteradores: esgotam depois de consumidos

# ── Closure ──
def externa(x):
    def interna():
        return x          # captura x
    return interna

def contador():
    n = 0
    def inc():
        nonlocal n        # para ESCREVER na variável capturada
        n += 1
        return n
    return inc

# ⚠️ Armadilha do laço: captura a VARIÁVEL, não o valor
[lambda: i for i in range(3)]        # ❌ todos devolvem 2
[lambda i=i: i for i in range(3)]    # ✅

# ── Decorador ──
import functools

def decorador(funcao):
    @functools.wraps(funcao)         # ⚠️ NUNCA esqueça
    def envelope(*args, **kwargs):
        # antes
        r = funcao(*args, **kwargs)
        # depois
        return r
    return envelope

def decorador_com_parametro(param):  # três níveis
    def real(funcao):
        @functools.wraps(funcao)
        def envelope(*a, **k):
            return funcao(*a, **k)
        return envelope
    return real

@a
@b
def f(): ...          # ≡ a(b(f)) — aplicam de baixo para cima

# ── functools ──
@functools.lru_cache(maxsize=128)    # só funções PURAS, args hasheáveis
@functools.cache                     # 3.9+, = lru_cache(None)
funcao.cache_info()  funcao.cache_clear()
partial(funcao, arg_fixo=valor)
```

## ✅ Checklist de saída

- [ ] Sei a ordem obrigatória dos parâmetros e o que o `*` sozinho faz
- [ ] Uso `*`/`**` tanto para coletar quanto para espalhar
- [ ] Escrevo `def envelope(*args, **kwargs)` para repasse genérico
- [ ] Entendo que função é objeto, e não confundo `f` com `f()`
- [ ] Substituo cadeias de `if/elif` por dicionário de despacho
- [ ] Uso `lambda` só onde cabe, e nunca atribuo a um nome
- [ ] Ordeno por múltiplos critérios com tupla no `key=`
- [ ] Sei que `map`/`filter` devolvem iteradores que esgotam
- [ ] Sempre passo o valor inicial no `reduce`
- [ ] Explico o que é closure e uso `nonlocal` quando preciso
- [ ] Reconheço e corrijo a armadilha da closure em laço
- [ ] Escrevo fábricas de função para eliminar duplicação
- [ ] Escrevo decoradores, **sempre** com `@functools.wraps`
- [ ] Sei escrever decorador com parâmetro (três níveis)
- [ ] Sei que decoradores empilhados aplicam de baixo para cima
- [ ] Uso `lru_cache` apenas em funções puras
- [ ] Uso `partial` para especializar funções

---

### ➡️ Próxima aula

**`04_02_Iteradores_e_Geradores.ipynb`** — O protocolo de iteração e o `yield`. Como processar um arquivo de 10 GB usando 10 MB de memória.